# Battery Swelling Prediction Workflow

This notebook records the end-to-end workflow for:

1. dataset cleaning / subset preparation
2. frequency-domain ECM fitting
3. time-domain ECM fitting
4. model training
5. evaluation and plotting

It is designed to reuse the existing command-line scripts in this repo, while making the run history and outputs easier to inspect in one place.

## 0. Manual configuration

Set the output folder and dataset paths here before running anything else.

In [ ]:
from pathlib import Path
import os
import subprocess
import shlex
import json
import pandas as pd

REPO_ROOT = Path.cwd()

# Support running the notebook from the repo root, notebooks/, or src/.
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent

VENV_PYTHON = REPO_ROOT / ".venv" / "bin" / "python"

# -------------------------
# Manual run configuration
# -------------------------
RUN_NAME = "notebook_run"
OUTPUT_ROOT = REPO_ROOT / "data" / RUN_NAME
LOG_DIR = OUTPUT_ROOT / "logs"

# Generic dataset paths
WORKBOOK_XLSX_DIR = REPO_ROOT / "dataset" / "udc_xlsx"
RAW_DATA_DIR = REPO_ROOT / "dataset" / "raw_data"

# Example real local paths for this repo (edit as needed)
# WORKBOOK_XLSX_DIR = REPO_ROOT / "dataset" / "OneDrive_1_2-20-2026" / "HYCL"
# RAW_DATA_DIR = REPO_ROOT / "dataset" / "raw_data" / "HYCL"

# Frequency-domain fitting configuration
ECM_OUT_DIR = OUTPUT_ROOT / "ecm_w_cycle_td_compatible"
PRIOR_CSV = OUTPUT_ROOT / "eis_time_domain_priors.csv"
RAW_FEATURE_CSV = OUTPUT_ROOT / "device_ecm_with_priors.csv"
MERGED_FEATURE_TABLE = OUTPUT_ROOT / "feature_table_with_device.csv"
TRAIN_TABLE = OUTPUT_ROOT / "feature_table_train.csv"
MODEL_OUT_DIR = OUTPUT_ROOT / "results_xgb"

# Optional: if full frequency-domain fitting is too heavy, point this to a small seed subset
WORKBOOK_PRIOR_SEED_DIR = WORKBOOK_XLSX_DIR

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)
print("repo_root =", REPO_ROOT)
print("venv_python =", VENV_PYTHON)
OUTPUT_ROOT


In [ ]:
def run(cmd, log_name=None, cwd=REPO_ROOT, check=True):
    print(cmd)
    if log_name is not None:
        log_path = LOG_DIR / log_name
        with open(log_path, "w", encoding="utf-8") as f:
            proc = subprocess.run(cmd, shell=True, cwd=cwd, stdout=f, stderr=subprocess.STDOUT, text=True)
        print(f"[log] {log_path}")
    else:
        proc = subprocess.run(cmd, shell=True, cwd=cwd, text=True)
    if check and proc.returncode != 0:
        raise RuntimeError(f"command failed with code {proc.returncode}: {cmd}")
    return proc.returncode


def show_log(log_name, n=200):
    p = LOG_DIR / log_name
    text = p.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    print("\n".join(lines[-n:]))


## 1. Dataset cleaning / subset preparation

This section checks the workbook dataset and raw dataset, and optionally builds a raw subset that is easier to process.

Typical reasons to subset raw data:

- one representative file per serial
- only serials that appear in the labeled workbook-derived feature table
- only files that overlap with ECM priors

In [ ]:
print('Workbook dir:', WORKBOOK_XLSX_DIR)
print('Raw dir:', RAW_DATA_DIR)
print('Workbook exists:', WORKBOOK_XLSX_DIR.exists())
print('Raw exists:', RAW_DATA_DIR.exists())

xlsx_files = sorted(WORKBOOK_XLSX_DIR.rglob('*.xlsx')) if WORKBOOK_XLSX_DIR.exists() else []
raw_files = sorted([p for p in RAW_DATA_DIR.rglob('*') if p.is_file()]) if RAW_DATA_DIR.exists() else []
print('n workbook files =', len(xlsx_files))
print('n raw files =', len(raw_files))

### 1a. Optional: create a one-file-per-serial raw subset

This is often useful for notebook-scale experiments when the full raw dataset is too large.

In [ ]:
RAW_SUBSET_DIR = OUTPUT_ROOT / "raw_subset_one_per_serial"

cmd = f'''{VENV_PYTHON} - <<"PY"
import sys, shutil
from pathlib import Path
import pandas as pd
sys.path.insert(0, 'src')
from parse_raw_maccor import parse_metadata

repo = Path('.').resolve()
raw_dir = Path(r"{RAW_DATA_DIR}")
out_dir = Path(r"{RAW_SUBSET_DIR}")
out_dir.mkdir(parents=True, exist_ok=True)

# If a feature table already exists elsewhere, you can use it to restrict serials.
# Otherwise this block simply keeps one file per serial across the raw dataset.
by_serial = {{}}
for p in sorted(raw_dir.iterdir()):
    if not p.is_file() or p.name.startswith('.'):
        continue
    serial = str(parse_metadata('', p.name).get('serial','')).strip().upper()
    if serial and serial not in by_serial:
        by_serial[serial] = p

for serial, p in sorted(by_serial.items()):
    shutil.copy2(p, out_dir / p.name)

print('copied_files', len(by_serial))
print('copied_serials', len(by_serial))
PY'''
run(cmd, log_name='01_make_raw_subset.log')
show_log('01_make_raw_subset.log', n=80)

## 2. Frequency-domain ECM fitting

This step fits ECM parameters from workbook EIS sheets. The recommended branch for the time-domain workflow is the `td_compatible` circuit family.

If the full workbook dataset is too large, point `WORKBOOK_PRIOR_SEED_DIR` to a smaller seed subset first.

In [ ]:
cmd = f'''MPLCONFIGDIR="{OUTPUT_ROOT / '.mplconfig'}" {VENV_PYTHON} src/ecm_fit.py   --xlsx_dir "{WORKBOOK_PRIOR_SEED_DIR}"   --recursive   --sheet 02_PreEIS   --fit_mode best_block   --circuit_family td_compatible   --warburg none   --out_dir "{ECM_OUT_DIR}"   --log_file "{LOG_DIR / '02_ecm_fit.internal.log'}"'''
run(cmd, log_name='02_ecm_fit.log')
show_log('02_ecm_fit.log', n=120)

## 3. Convert frequency-domain ECM into time-domain priors

This step converts frequency-domain ECM results into:

- initial guesses
- lower bounds
- upper bounds

for the time-domain fitter.

In [ ]:
cmd = f'''{VENV_PYTHON} src/build_eis_time_domain_priors.py   --ecm_dir "{ECM_OUT_DIR}"   --out_csv "{PRIOR_CSV}"'''
run(cmd, log_name='03_build_priors.log')
show_log('03_build_priors.log', n=120)

priors = pd.read_csv(PRIOR_CSV)
priors.head()

## 4. Time-domain ECM fitting from raw device data

This step uses raw `current / voltage / time` signals to extract device-side ECM-inspired features.

Common modes:

- `prior_mode = global`: ignore per-battery serial matching and use global prior bounds
- `prior_mode = serial_cycle`: use serial + cycle aligned priors when available

In [ ]:
RAW_DIR_FOR_FIT = RAW_SUBSET_DIR if RAW_SUBSET_DIR.exists() else RAW_DATA_DIR

cmd = f'''{VENV_PYTHON} src/extract_device_ecm_features.py   --raw_dir "{RAW_DIR_FOR_FIT}"   --eis_prior_csv "{PRIOR_CSV}"   --prior_mode global   --prior_align_mode last_le   --fit_mode td_only   --cycle_mode last   --out_csv "{RAW_FEATURE_CSV}"'''
run(cmd, log_name='04_extract_device_ecm.log')
show_log('04_extract_device_ecm.log', n=120)

device_df = pd.read_csv(RAW_FEATURE_CSV)
device_df.head()

## 5. Build / merge the training table

This section assumes you have a labeled feature table or want to build one from workbook data.

If you already have a workbook-derived feature table, point `BASE_FEATURE_TABLE` to it.
Otherwise, run `build_feature_table.py` first.

In [ ]:
BASE_FEATURE_TABLE = OUTPUT_ROOT / "feature_table_base.csv"

cmd = f'''{VENV_PYTHON} src/build_feature_table.py   --xlsx_dir "{WORKBOOK_XLSX_DIR}"   --ecm_dir "{ECM_OUT_DIR}"   --out_csv "{BASE_FEATURE_TABLE}"   --min_cycle 5   --max_cycle 200   --future_k 20   --soc_target 50   --dcir_align_mode last_le'''
run(cmd, log_name='05_build_feature_table.log')
show_log('05_build_feature_table.log', n=120)

In [ ]:
cmd = f'''{VENV_PYTHON} src/extract_device_ecm_features.py   --raw_dir "{RAW_DIR_FOR_FIT}"   --eis_prior_csv "{PRIOR_CSV}"   --prior_mode global   --prior_align_mode last_le   --fit_mode td_only   --cycle_mode last   --out_csv "{RAW_FEATURE_CSV}"   --feature_table_csv "{BASE_FEATURE_TABLE}"   --out_feature_table_csv "{MERGED_FEATURE_TABLE}"   --align_mode last_le'''
run(cmd, log_name='06_merge_device_features.log')
show_log('06_merge_device_features.log', n=120)

merged_df = pd.read_csv(MERGED_FEATURE_TABLE)
merged_df.head()

### 5a. Create the final training table used for a device-oriented model

This example keeps:

- cycle / capacity / DCIR features
- time-domain ECM features

and excludes lab-side frequency-domain ECM features from the model inputs.

In [ ]:
need = [
    'feat_cycle_t',
    'feat_capacity_t',
    'feat_capacity_slope_10',
    'feat_dcir_soc_t',
    'feat_dev_r0_proxy_ohm',
    'feat_dev_td_R_diff_total_ohm',
    'feat_dev_td_R_total_proxy_ohm',
]

train_df = merged_df.dropna(subset=[c for c in need if c in merged_df.columns]).copy()
train_df.to_csv(TRAIN_TABLE, index=False)
print('saved', TRAIN_TABLE)
print(train_df.shape)
print('n_serials =', train_df['serial'].nunique() if 'serial' in train_df.columns else 'n/a')
print('n_cell_keys =', train_df['cell_key'].nunique() if 'cell_key' in train_df.columns else 'n/a')

## 6. Model training

This section trains a classical XGBoost model. You can adapt the same pattern for MLP/CNN/LSTM or Transformer models.

In [ ]:
CUSTOM_FEATURES = ','.join([
    'feat_cycle_t',
    'feat_capacity_t',
    'feat_capacity_slope_10',
    'feat_dcir_soc_t',
    'feat_dev_r0_proxy_ohm',
    'feat_dev_td_R_diff_total_ohm',
    'feat_dev_td_R_total_proxy_ohm',
])

cmd = f'''{VENV_PYTHON} src/train_swelling_models.py   --table_csv "{TRAIN_TABLE}"   --out_dir "{MODEL_OUT_DIR}"   --target_mode current   --sample_mode rowwise   --label_mode absolute   --target_transform log   --max_input_cycle 120   --models XGBoost   --feature_set custom   --custom_features "{CUSTOM_FEATURES}"   --xgb_n_estimators 1200   --xgb_max_depth 4   --xgb_learning_rate 0.015   --xgb_subsample 0.85   --xgb_colsample_bytree 0.85   --xgb_min_child_weight 2   --xgb_reg_alpha 0.05   --xgb_reg_lambda 2.0   --run_tag notebook_xgb   --log_file "{LOG_DIR / '07_train_xgb.internal.log'}"'''
run(cmd, log_name='07_train_xgb.log')
show_log('07_train_xgb.log', n=120)

In [ ]:
result_csvs = sorted(MODEL_OUT_DIR.glob('results__*.csv'))
pred_csvs = sorted(MODEL_OUT_DIR.glob('predictions__*.csv'))
print('results:', result_csvs)
print('predictions:', pred_csvs)
if result_csvs:
    pd.read_csv(result_csvs[-1])

## 7. Evaluation and plotting

This step generates:

- prediction vs truth scatter
- permutation importance
- correlation matrix

using the bundled feature-analysis runner.

In [ ]:
cmd = f'''{VENV_PYTHON} src/run_feature_analysis.py   --table_csv "{TRAIN_TABLE}"   --results_dir "{MODEL_OUT_DIR}"   --out_dir "{MODEL_OUT_DIR}"   --custom_features "{CUSTOM_FEATURES}"   --group_tag HYCL   --target_mode current   --sample_mode rowwise   --label_mode absolute   --target_transform log   --max_input_cycle 120   --model XGBoost   --metric mae   --n_repeats 30   --corr_method spearman   --xgb_n_estimators 1200   --xgb_max_depth 4   --xgb_learning_rate 0.015   --xgb_subsample 0.85   --xgb_colsample_bytree 0.85   --xgb_min_child_weight 2   --xgb_reg_alpha 0.05   --xgb_reg_lambda 2.0'''
run(cmd, log_name='08_feature_analysis.log')
show_log('08_feature_analysis.log', n=120)

In [ ]:
from IPython.display import Image, display

for name in [
    'pred_scatter__combined.png',
    'feature_corr__features_targets.png',
    'perm_importance__current__absolute__current_cycle__HYCL__XGBoost__mae.png',
]:
    p = MODEL_OUT_DIR / name
    if p.exists():
        print(p)
        display(Image(filename=str(p)))

## 8. Optional: compare frequency-domain and time-domain ECM ranges

If you want to compare prior ranges and fitted time-domain ranges, this section can be filled with the range-plot scripts already used in the repo.

In [ ]:
# Example placeholder for custom range-plot logic.
# You can adapt the same plotting pattern used for:
# - ecm_range__frequency_domain_prior_bounds__matched5.png
# - ecm_range__time_domain_fitted_ranges__matched5.png
# - ecm_range__freq_vs_time__matched5.png
print('Add your matched range-plot code here if this run requires it.')

## 9. Final notes

Recommended things to record after each run:

- dataset paths used
- whether the run is hybrid or device-oriented
- prior mode (`global`, `serial_cycle`, etc.)
- final feature list
- final `MAE / RMSE`
- key plots
- any failure modes such as `insufficient_points`